# Latent-Confounded Coordination Matrix Game — Evaluation

Rebuttal experiment: **why reward-prediction accuracy is not a valid causal-credit yardstick**,
and whether a trained proximal bridge recovers the population-identified response.

This notebook imports `latent_context_matrix_game.py` from the same directory. The module owns
the DGP, population oracles, trained estimators, and exact population-MSE calculation; this
notebook only orchestrates comparisons, sweeps, and plots.

**Four estimators (2×2: identification × estimation)**

1. `fit_bridge_oracle` — exact population proximal bridge $h(a,w,x)$
2. `fit_bridge_mlp` — bridge MLP trained with the Torch `r_pci.py` objective
   $r^\top K(\gamma I+\lambda K)^{-1}r/B$
3. `fit_obs_regression_oracle` — exact population observational $E[R\mid A_1,W,X]$
4. `fit_reward_mlp` — MSE-trained observational regressor

The two population oracles isolate identification. The two MLPs can either be run under a
matched protocol or under tuned estimator-specific hyperparameters. In the tuned cells below,
`steps` and `batch` are intentionally shared, while `lr`, `hidden`, and bridge-only regularization
use the credit-error sweep winners.

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

import latent_context_matrix_game as G   # the reference module (.py in same dir)

# reproducibility / speed knobs for the notebook
import torch
torch.set_num_threads(4)

np.set_printoptions(precision=3, suppress=True)
print("module loaded:", G.__name__)

## 1. Sanity check — assumptions & closed-form ground truth

Confirm the toy actually satisfies the PCI assumptions (conditional independencies +
positivity), and that the closed-form `true_tau` behaves as designed:
$\tau(x)=\delta x$, **independent of the confounding strength $\beta$**.


In [ ]:
cfg = G.GameConfig(beta=1.0, delta=1.0, gamma=0.5, q=0.85, p_w=0.85, eps=0.05)
data = G.sample_dataset(cfg, n=50_000, seed=1)

G.verify_assumptions(data)

print("\nTrue interventional credit  tau(x) = delta * x  (independent of beta):")
for x in (-1, 1):
    print(f"  tau({x:+d}) = {G.true_tau(cfg, x):+.3f}")
print(f"  marginal E_X[tau] = {G.true_marginal_tau(cfg):+.3f}  (should be 0)")

## 2. Symmetric population-oracle comparison (core result)

Both estimators below enumerate the known DGP exactly, so their gap is purely an identification
gap. The bridge recovers $\tau(x)$ exactly, while observational regression retains confounding
bias. Reward-prediction MSE is computed exactly over the population, including reward-noise
variance.

In [ ]:
h = G.fit_bridge_oracle(cfg)
g = G.fit_obs_regression_oracle(cfg)

print("tau(x): true / bridge-oracle / obs-regression-oracle")
for x in (-1, 1):
    target = G.true_tau(cfg, x)
    bridge = G.bridge_do_response(h, 1, x) - G.bridge_do_response(h, 0, x)
    obs = (G.obs_regression_do_response(g, 1, x)
           - G.obs_regression_do_response(g, 0, x))
    print(f"  x={x:+d} | true={target:+.3f}  bridge={bridge:+.3f}  obs-oracle={obs:+.3f}")

print(f"\npopulation R-prediction MSE: bridge={G._pred_mse_table(h, cfg):.4f}   "
      f"obs-oracle={G._pred_mse_table(g, cfg):.4f}")
print("=> The bridge predicts R worse yet recovers causal credit exactly.")

### 2b. Estimation-level check — matched-protocol MLPs

The bridge MLP minimizes the same regularized kernel objective used in `models/torch/r_pci.py`:

$$
\frac{1}{B}r^\top K(\gamma_{tikh}I+\lambda_{st}K)^{-1}r.
$$

The observational MLP minimizes minibatch MSE. Both use the same network, initial weights,
optimizer, learning rate, batch indices, batch size, and number of updates. Consequently this
comparison changes only the objective. Set `RUN_MLP=True` to run five training seeds.

In [ ]:
RUN_MLP = True  # set True to train both neural nets

# Credit-error tuned settings. steps and batch are shared across the two estimators.
COMMON_TRAIN_KWARGS = dict(
    steps=1000,
    batch=512,
)
OBS_ESTIMATOR_KWARGS = dict(
    lr=3e-3,
    hidden=64,
)
BRIDGE_ESTIMATOR_KWARGS = dict(
    lr=3e-3,
    hidden=64,
    tikhonov_gamma=1e-4,
    stabilizer_lambda=1.0,
    kernel_mode="ours",   # or "ours"
    sigma2 = 1.0
)

OBS_TRAIN_KWARGS = dict(COMMON_TRAIN_KWARGS, **OBS_ESTIMATOR_KWARGS)
BRIDGE_TRAIN_KWARGS = dict(COMMON_TRAIN_KWARGS, **BRIDGE_ESTIMATOR_KWARGS)

if RUN_MLP:
    print("regularized bridge loss on the same evaluation batch:")
    print(f"  bridge oracle          : {G.bridge_loss_at(h, data, is_table=True):+.6f}")
    print(f"  obs-regression oracle  : {G.bridge_loss_at(g, data, is_table=True):+.6f}")
    print("\ntraining kwargs:")
    print(f"  common : {COMMON_TRAIN_KWARGS}")
    print(f"  bridge : {BRIDGE_ESTIMATOR_KWARGS}")
    print(f"  obs    : {OBS_ESTIMATOR_KWARGS}")

    SEEDS = [42, 17, 89, 33, 56, 71, 4, 92, 15, 68]
    bridge_errs, obs_errs = [], []
    bridge_mses, obs_mses = [], []
    for s in SEEDS:
        bmlp = G.fit_bridge_mlp(data, seed=s, **BRIDGE_TRAIN_KWARGS)
        omlp = G.fit_reward_mlp(data, seed=s, **OBS_TRAIN_KWARGS)
        bridge_err = max(
            abs((G.mlp_do_response(bmlp, 1, x) - G.mlp_do_response(bmlp, 0, x))
                - G.true_tau(cfg, x))
            for x in (-1, 1)
        )
        obs_err = max(
            abs((G.mlp_do_response(omlp, 1, x) - G.mlp_do_response(omlp, 0, x))
                - G.true_tau(cfg, x))
            for x in (-1, 1)
        )
        bridge_errs.append(bridge_err)
        obs_errs.append(obs_err)
        bridge_mses.append(G._pred_mse_mlp(bmlp, cfg))
        obs_mses.append(G._pred_mse_mlp(omlp, cfg))
        print(f"  seed {s}: bridge err={bridge_err:.4f}  obs err={obs_err:.4f}")

    bridge_errs = np.asarray(bridge_errs)
    obs_errs = np.asarray(obs_errs)
    bridge_oracle_err = max(
        abs(G.bridge_do_response(h, 1, x) - G.bridge_do_response(h, 0, x)
            - G.true_tau(cfg, x))
        for x in (-1, 1)
    )
    obs_oracle_err = max(
        abs(G.obs_regression_do_response(g, 1, x)
            - G.obs_regression_do_response(g, 0, x)
            - G.true_tau(cfg, x))
        for x in (-1, 1)
    )

    print(f"\n{'':30}{'population oracle':>20}{'learned (mean±sd)':>22}")
    print("-" * 72)
    print(f"{'proximal bridge':30}{bridge_oracle_err:>20.4f}"
          f"{bridge_errs.mean():>14.4f} ± {bridge_errs.std():.4f}")
    print(f"{'observational':30}{obs_oracle_err:>20.4f}"
          f"{obs_errs.mean():>14.4f} ± {obs_errs.std():.4f}")
    print(f"\npopulation prediction MSE (mean): bridge={np.mean(bridge_mses):.4f}  "
          f"observational={np.mean(obs_mses):.4f}")
else:
    print("RUN_MLP=False — skipping tuned neural-net training.")


### 2c. Kernel ablation — `r_pci` blocks vs. configurable legacy RBF

This ablation changes only the conditioning kernel $K$ in the regularized bridge objective.

Set `LEGACY_SIGMA2` in the next cell to any positive value you want to test.

- `kernel_mode="r_pci"`: standardized product-RBF blocks for $(Z,A_1,X)$
- `kernel_mode="legacy_rbf", sigma2=LEGACY_SIGMA2`: isotropic
  $\exp(-\lVert c-c'\rVert^2/(2\sigma^2))$ on the natural encoding

The network, initialization, minibatches, optimizer, learning rate, number of updates,
Tikhonov regularization, and stabilizer are identical. Thus this section isolates bandwidth and
kernel geometry without reverting to the previous U-statistic objective.


In [ ]:
RUN_KERNEL_COMPARISON = False  # five seeds; set False to skip the kernel ablation
SIGMA2 = 1.0          # RBF bandwidth

if RUN_KERNEL_COMPARISON:
    KERNEL_SEEDS = [0, 1, 2, 3, 4]
    kernel_results = {"r_pci": [], "legacy_rbf": []}
    kernel_mses = {"r_pci": [], "legacy_rbf": []}

    print("bridge-oracle loss under each evaluation kernel:")
    print(f"  r_pci blocks              : {G.bridge_loss_at(h, data, is_table=True):+.6f}")
    print(f"  legacy sigma2={SIGMA2:g} : "
          f"{G.bridge_loss_at(h, data, is_table=True, kernel_mode='legacy_rbf', sigma2=SIGMA2):+.6f}")

    for seed in KERNEL_SEEDS:
        models = {
            "r_pci": G.fit_bridge_mlp(
                data, seed=seed, **BRIDGE_TRAIN_KWARGS
            ),
            "legacy_rbf": G.fit_bridge_mlp(
                data, seed=seed,
                **dict(BRIDGE_TRAIN_KWARGS, kernel_mode="legacy_rbf", sigma2=SIGMA2)
            ),
        }
        line = []
        for name, model in models.items():
            error = max(
                abs((G.mlp_do_response(model, 1, x) - G.mlp_do_response(model, 0, x))
                    - G.true_tau(cfg, x))
                for x in (-1, 1)
            )
            kernel_results[name].append(error)
            kernel_mses[name].append(G._pred_mse_mlp(model, cfg))
            line.append(f"{name}={error:.4f}")
        print(f"  seed {seed}: " + "  ".join(line))

    for name in kernel_results:
        kernel_results[name] = np.asarray(kernel_results[name])
        kernel_mses[name] = np.asarray(kernel_mses[name])

    print("\nmean ± sd")
    for name in ("r_pci", "legacy_rbf"):
        errors = kernel_results[name]
        mses = kernel_mses[name]
        print(f"  {name:10s} credit={errors.mean():.4f}±{errors.std():.4f}  "
              f"population MSE={mses.mean():.4f}±{mses.std():.4f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.5, 3.8))
    names = ["r_pci", "legacy_rbf"]
    labels = ["r_pci blocks", rf"legacy $\sigma^2={SIGMA2:g}$"]
    x_pos = np.arange(2)
    ax1.bar(x_pos, [kernel_results[n].mean() for n in names],
            yerr=[kernel_results[n].std() for n in names], capsize=4,
            color=["#00aff0", "#0057b7"])
    ax1.axhline(0.399, color="#c1272d", linestyle="--", alpha=0.7,
                label="obs. oracle error")
    ax1.set_xticks(x_pos, labels)
    ax1.set_ylabel(r"credit error $\max_x|\hat\delta-\delta|$")
    ax1.set_title("Bridge credit error")
    ax1.legend(fontsize=8); ax1.grid(axis="y", alpha=0.3)

    ax2.bar(x_pos, [kernel_mses[n].mean() for n in names],
            yerr=[kernel_mses[n].std() for n in names], capsize=4,
            color=["#00aff0", "#0057b7"])
    ax2.set_xticks(x_pos, labels)
    ax2.set_ylabel("population reward MSE")
    ax2.set_title("Bridge prediction error")
    ax2.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("RUN_KERNEL_COMPARISON=False — skipping kernel ablation.")


### 2d. Sample-size sweep — does the learned bridge approach its population oracle?

The population-oracle errors do not depend on sample size. The learned estimators are retrained
at each $n$ with the same step schedule for both objectives. If the bridge's estimation error
shrinks while the observational gap remains, the remaining observational error is an
identification failure rather than an optimization artifact.

In [ ]:
RUN_N_SWEEP = False  # set True to train 2 nets × len(Ns) × len(SEEDS_N)

# How many optimizer updates to use at each dataset size.
# - "sqrt"   : max(N_STEP_MIN, int(N_STEP_BASE * sqrt(n / N_STEP_REF_N)))
# - "linear" : max(N_STEP_MIN, int(N_STEP_BASE * n / N_STEP_REF_N))
N_STEP_SCALING = "linear"
N_STEP_BASE = COMMON_TRAIN_KWARGS["steps"]
N_STEP_REF_N = 50_000
N_STEP_MIN = 1000


def n_sweep_steps(n: int) -> int:
    scale = n / N_STEP_REF_N
    if N_STEP_SCALING == "sqrt":
        return max(N_STEP_MIN, int(N_STEP_BASE * np.sqrt(scale)))
    if N_STEP_SCALING == "linear":
        return max(N_STEP_MIN, int(N_STEP_BASE * scale))
    raise ValueError(f"Unsupported N_STEP_SCALING: {N_STEP_SCALING}")


if RUN_N_SWEEP:
    Ns = [2_000, 5_000, 10_000, 20_000, 50_000, 100_000]
    SEEDS_N = [0, 1, 2, 3, 4]
    print(f"sample-size sweep step scaling: {N_STEP_SCALING}")
    bridge_oracle_err = max(
        abs(G.bridge_do_response(h, 1, x) - G.bridge_do_response(h, 0, x)
            - G.true_tau(cfg, x))
        for x in (-1, 1)
    )
    obs_oracle_err = max(
        abs(G.obs_regression_do_response(g, 1, x)
            - G.obs_regression_do_response(g, 0, x)
            - G.true_tau(cfg, x))
        for x in (-1, 1)
    )
    n_rows = []
    for n in Ns:
        d_n = G.sample_dataset(cfg, n=n, seed=1)
        steps = n_sweep_steps(n)
        shared = dict(steps=steps, batch=COMMON_TRAIN_KWARGS["batch"])
        bridge_kwargs = dict(shared, **BRIDGE_ESTIMATOR_KWARGS)
        obs_kwargs = dict(shared, **OBS_ESTIMATOR_KWARGS)
        bridge_errors, obs_errors = [], []
        for s in SEEDS_N:
            bmlp = G.fit_bridge_mlp(d_n, seed=s, **bridge_kwargs)
            omlp = G.fit_reward_mlp(d_n, seed=s, **obs_kwargs)
            bridge_errors.append(max(
                abs((G.mlp_do_response(bmlp, 1, x) - G.mlp_do_response(bmlp, 0, x))
                    - G.true_tau(cfg, x))
                for x in (-1, 1)
            ))
            obs_errors.append(max(
                abs((G.mlp_do_response(omlp, 1, x) - G.mlp_do_response(omlp, 0, x))
                    - G.true_tau(cfg, x))
                for x in (-1, 1)
            ))
        n_rows.append(dict(
            n=n,
            steps=steps,
            step_scaling=N_STEP_SCALING,
            orc_b=bridge_oracle_err,
            orc_o=obs_oracle_err,
            lrn_b=float(np.mean(bridge_errors)),
            lrn_b_sd=float(np.std(bridge_errors)),
            lrn_o=float(np.mean(obs_errors)),
            lrn_o_sd=float(np.std(obs_errors)),
        ))
        row = n_rows[-1]
        print(
            f"n={n:>7,} steps={steps:>5}  "
            f"bridge: oracle={row['orc_b']:.4f} learned={row['lrn_b']:.4f}±{row['lrn_b_sd']:.4f}  "
            f"obs: oracle={row['orc_o']:.4f} learned={row['lrn_o']:.4f}±{row['lrn_o_sd']:.4f}"
        )
else:
    print("RUN_N_SWEEP=False — skipping sample-size sweep.")


In [ ]:
if RUN_N_SWEEP:
    fig, ax = plt.subplots(figsize=(6.5, 4.4))
    ns = [r["n"] for r in n_rows]
    ax.plot(ns, [r["orc_b"] for r in n_rows], "o-", color="#0057b7", alpha=0.6,
            label="Proximal, population oracle")
    ax.errorbar(ns, [r["lrn_b"] for r in n_rows], yerr=[r["lrn_b_sd"] for r in n_rows],
                fmt="^--", color="#0057b7", label="Proximal, learned (regularized kernel)")
    ax.plot(ns, [r["orc_o"] for r in n_rows], "o-", color="#c1272d", alpha=0.6,
            label="Observational, population oracle")
    ax.errorbar(ns, [r["lrn_o"] for r in n_rows], yerr=[r["lrn_o_sd"] for r in n_rows],
                fmt="^--", color="#c1272d", label="Observational, learned (MSE)")
    ax.set_xscale("log")
    ax.set_xlabel("sample size n")
    ax.set_ylabel(r"credit error  $\max_x|\hat\delta(x)-\delta(x)|$")
    ax.set_title("Learned estimators vs population oracles")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig("n_sweep.png", dpi=150); plt.show()

## 3. Confounding sweep over $\beta$

The decisive figure. True credit is flat in $\beta$ (a horizontal line at $\delta$); the
bridge tracks it, the observational oracle diverges **linearly** in $\beta$. The curves
meet at $\beta=0$ — the weak-confounding regime that explains why a plain MLP looked fine
on the diagnostic Stag-Hunt domain.


In [ ]:
RUN_MLPS_IN_SWEEP = True  # trains both estimator-specific MLPs at every beta

sweep_beta = G.run_sweep(
    "beta", np.linspace(0.0, 2.0, 11),
    base=dict(delta=1.0, gamma=0.5, q=0.85, p_w=0.85, eps=0.05),
    n=120_000,
    with_mlp=RUN_MLPS_IN_SWEEP,
    with_bridge_mlp=RUN_MLPS_IN_SWEEP,
    mlp_kwargs=OBS_TRAIN_KWARGS,
    bridge_kwargs=BRIDGE_ESTIMATOR_KWARGS,
)


In [ ]:
# setting plt global config
plt.rc("font", size=14)
plt.rc("axes", labelsize=14)
plt.rc("xtick", labelsize=14)
plt.rc("ytick", labelsize=14)
plt.rc("legend", fontsize=12)

In [ ]:
vals = sweep_beta["values"]; rows = sweep_beta["rows"]
has_mlp = rows[0].get("tau_err_mlp") is not None
has_bridge_mlp = rows[0].get("tau_err_bridge_mlp") is not None

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))

ax1.plot(vals, [r["tau_err_obs"] for r in rows], "o-",
         label="Obs. regression (oracle)", color="#c1272d")
if has_mlp:
    ax1.plot(vals, [r["tau_err_mlp"] for r in rows], "^--",
             label="Obs. regression (learned)", color="#c1272d", alpha=0.8)
ax1.plot(vals, [r["tau_err_bridge"] for r in rows], "o-",
         label="Proximal bridge (oracle)", color="#0057b7")
if has_bridge_mlp:
    ax1.plot(vals, [r["tau_err_bridge_mlp"] for r in rows], "^--",
             label="Proximal bridge (learned)", color="#0057b7", alpha=0.8)
ax1.set_xlabel(r"unobserved-confounding strength  $\beta$")
ax1.set_ylabel(r"credit error  $\max_x|\hat\delta(x)-\delta(x)|$")
ax1.set_title("Credit error vs confounding")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(vals, [r["mse_obs"] for r in rows], "o-",
         label="Obs. regression (oracle)", color="#c1272d")
if has_mlp:
    ax2.plot(vals, [r["mse_mlp"] for r in rows], "^--",
             label="Obs. regression (learned)", color="#c1272d", alpha=0.8)
ax2.plot(vals, [r["mse_bridge"] for r in rows], "o-",
         label="Proximal bridge (oracle)", color="#0057b7")
if has_bridge_mlp:
    ax2.plot(vals, [r["mse_bridge_mlp"] for r in rows], "^--",
             label="Proximal bridge (learned)", color="#0057b7", alpha=0.8)
ax2.set_xlabel(r"unobserved-confounding strength  $\beta$")
ax2.set_ylabel("population reward-prediction MSE")
ax2.set_title("R-prediction MSE")
ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 4. Proxy-noise sweep over $p_w$

Second knob: as the outcome proxy $W$ gets noisier ($p_w \to 1/2$), simply conditioning on
$W$ (what the regression does) fails to block the backdoor, so its bias grows. The bridge
stays robust as long as completeness holds. At $p_w=1$ ($W$ deterministic given the latent)
the gap closes.


In [ ]:
sweep_pw = G.run_sweep(
    "p_w", np.linspace(0.55, 1.0, 10),
    base=dict(beta=1.0, delta=1.0, gamma=0.5, q=0.85, eps=0.05),
    with_mlp=True,
    with_bridge_mlp=True,
    mlp_kwargs=OBS_TRAIN_KWARGS,
    bridge_kwargs=BRIDGE_ESTIMATOR_KWARGS,
)

vals = sweep_pw["values"]; rows = sweep_pw["rows"]
has_mlp = rows[0].get("tau_err_mlp") is not None
has_bridge_mlp = rows[0].get("tau_err_bridge_mlp") is not None

fig, ax = plt.subplots(figsize=(6, 4.4))
ax.plot(vals, [r["tau_err_obs"] for r in rows], "o-",
        label="Obs. regression (oracle)", color="#c1272d")
if has_mlp:
    ax.plot(vals, [r["tau_err_mlp"] for r in rows], "^--",
            label="Obs. regression (learned)", color="#c1272d", alpha=0.8)
ax.plot(vals, [r["tau_err_bridge"] for r in rows], "o-",
        label="Proximal bridge (oracle)", color="#0057b7")
if has_bridge_mlp:
    ax.plot(vals, [r["tau_err_bridge_mlp"] for r in rows], "^--",
            label="Proximal bridge (learned)", color="#0057b7", alpha=0.8)
ax.set_xlabel(r"outcome proxy fidelity  $p_w=P(W=U)$")
ax.set_ylabel(r"credit error  $\max_x|\hat\delta(x)-\delta(x)|$")
ax.set_title("Credit error vs proxy noise") 
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Per-state counterfactual baseline

The paper's baseline (p.7): $b_i(x,\tau^i)=\sum_a \tilde\pi^i(a\mid\tau^i)\,E[R\mid do(A_1=a),X=x]$.
Here `pi_tilde` is stubbed as the empirical $P(A_1=a\mid Z)$ — **replace with the paper's
renormalized policy** in the project codebase. Shown for both the bridge and the
observational oracle so the downstream credit-aligned reward can be compared.


In [ ]:
b_bridge = G.counterfactual_baseline(
    lambda a, x: G.bridge_do_response(h, a, x), data, cfg)
b_obs = G.counterfactual_baseline(
    lambda a, x: G.obs_regression_do_response(g, a, x), data, cfg)

print("counterfactual baseline b_i (first 8 samples):")
print("  bridge    :", np.round(b_bridge[:8], 3))
print("  obs-oracle:", np.round(b_obs[:8], 3))
print(f"\nmean |b_bridge-b_obs| = {np.abs(b_bridge-b_obs).mean():.4f}")

## 6. Notes for integration and reporting

- `fit_bridge_oracle` and `fit_obs_regression_oracle` enumerate the known DGP exactly. Their
  comparison is population-level identification, not a finite-sample cell-mean fit.
- `fit_bridge_mlp` always uses
  $r^\top K(\gamma I+\lambda K)^{-1}r/B$. The default `r_pci` kernel uses standardized
  product-RBF blocks for $(Z,A_1,X)$. The optional `legacy_rbf` mode uses isotropic
  $\sigma^2=\texttt{SIGMA2}$ on the natural encoding while keeping the current objective
  unchanged.
- The main MLP cells use credit-error tuned settings with shared `steps=1000` and `batch=512`.
  Estimator-specific knobs are `lr=3e-3, hidden=32` for observational MLP and
  `lr=3e-3, hidden=32, tikhonov_gamma=3e-3, stabilizer_lambda=1.0, kernel_mode="r_pci"`
  for bridge MLP.
- The sample-size sweep computes `steps` from `N_STEP_SCALING`: use `"sqrt"` for
  `max(1000, int(1000 * sqrt(n / 50_000)))` or `"linear"` for
  `max(1000, int(1000 * n / 50_000))`.
- The bridge MLP uses `lr=3e-3` across all notebook sweeps; this avoids the high-beta
  seed-specific instability observed with the faster `lr=1e-2` setting.
- `fit_bridge_mlp` and `fit_reward_mlp` receive the same model seed, number of updates, batch
  size, and minibatch index sequence. Learning rate, hidden size, and bridge-only regularization
  are tuned separately from the credit-error sweep.
- The exact hyperparameter sweep ranges and compact-final candidate list are recorded in
  `latent_context_matrix_game_hparam_sweep.md`.
- Prediction MSE is exact population MSE, including reward-noise variance.
- `pi_tilde` remains an empirical $P(A_1\mid Z)$ stub in §5. The main $\tau(x)$ plots do not
  depend on it, but the displayed counterfactual baselines do.
